# Deep Cuisine Transfer

### Text style transfer applied to recipes

**Team:** Relja Pesic, Jelena Lazovic

Rewriting a recipe's instructions in a different cuisine's voice — without changing what's actually being cooked.


## The problem

- Given a recipe's instructions written in one cuisine's style (Italian), rewrite them so they read like another cuisine's style (Indian) — **without changing what's actually being cooked**.
- This is a **text style transfer** problem: separate *how it's said* (style) from *what is said* (content), then recombine content with a new style.
- There is no parallel data — nobody wrote the same recipe twice, once per cuisine — so plain supervised translation is not possible.
- Based on the disentanglement approach from **"Deep Cuisine Transfer"**, itself building on [John et al., 2019 — *Disentangled Representation Learning for Non-Parallel Text Style Transfer*](https://aclanthology.org/P19-1041/).

## Dataset

- **Food.com recipes with search terms and tags** (Kaggle), downloaded on demand via `kagglehub` — no credentials required.
- Filtered down to recipes tagged **Italian** or **Indian** only (from `search_terms`).
- Raw pool is imbalanced: Italian recipes outnumber Indian ones **~2.6:1**.
- Final working set: **13,095 recipes**, balanced 1:1, split **8,637 / 1,851 / 1,851** (train / val / test).

## Class imbalance: which strategy?

Before committing to random undersampling, we compared it against oversampling, SMOTE, and class-weighting on a quick BoW + Logistic Regression cuisine classifier, scored by ROC-AUC on an untouched (still-imbalanced) test set.

**This is the actual generated figure from `01_preprocessing.ipynb`:**

![Class-imbalance strategy comparison](presentation_assets/imbalance_strategy_auc.png)

All four strategies land within ~0.005 ROC-AUC of doing nothing at all — the cuisines are simply easy to tell apart from word choice. We kept random undersampling anyway: it performs on par with the rest, and it also shrinks the dataset, which matters once the much heavier seq2seq model starts training on it.

## Dataset EDA

Real output from `01_preprocessing.ipynb` — class balance after outlier filtering, and the recipe-length distribution used to pick the sequence-length cutoff:

![Class balance and recipe length distribution](presentation_assets/class_balance_and_length.png)

## Preprocessing pipeline

1. Clean raw text: HTML unescaping, Unicode normalization, strip noisy boilerplate ("read more", "hit play", "www").
2. Tokenize with NLTK (`word_tokenize` / `sent_tokenize`), strip punctuation.
3. Filter outlier-length recipes (95th-percentile cutoff).
4. Build **two parallel token columns**:
   - `steps_tokens` — lightly cleaned, keeps stopwords → the **seq2seq reconstruction target** (needs fluent, natural text).
   - `steps_tokens_bow` — heavily cleaned (stopwords, punctuation, numbers, hyphens stripped) → the **bag-of-words content signal**.

## Turning words into numbers

- **Word2Vec** (gensim, 150-dim vectors) trained on `steps_tokens` — gives every word a meaning-bearing vector. Feeds the model's input.
- **CountVectorizer** (bag-of-words) fit on `steps_tokens_bow` — gives each recipe a "which content words appear" vector. Used as a training target and later as an evaluation signal.
- Both artifacts are **saved to disk** (`word2vec.wordvectors`, `bow_vectorizer.joblib`) and only rebuilt if missing, so a saved model checkpoint never goes stale against a different vocabulary.

## Model architecture

A **GRU sequence-to-sequence autoencoder**, with the encoder forced to split its summary into two halves:

- **Encoder**: reads a recipe → one hidden vector → split into a **style latent** (small) and a **content latent** (the rest).
- **Decoder**: takes style + content glued back together → regenerates the recipe word by word.

![Model architecture: encoder splits into style/content latents, four classifier heads, decoder recombines them](presentation_assets/model_architecture.png)

| Head | Input | Job | Effect |
|---|---|---|---|
| style_classifier | style latent | predict cuisine | style latent *must* encode cuisine |
| content_classifier | content latent | predict BoW words | content latent *must* encode recipe content |
| adversarial style clf | content latent | predict cuisine | encoder fights this → cuisine leaks *out* of content |
| adversarial content clf | style latent | predict BoW words | encoder fights this → content leaks *out* of style |

The adversarial heads are the key trick from the paper — trained to succeed at guessing the "wrong" thing, while the encoder is trained to make them fail. That's what actually forces style and content apart.

## Training

We noticed training loss kept improving while the model wasn't generalizing any better — a classic overfitting tell. Fix: checkpoint on **validation** reconstruction loss instead of train loss, add **early stopping** (patience = 5 epochs), and weight decay.

Real training curves from `02_model.ipynb`:

![Training curves: reconstruction loss and adversarial probe accuracy](presentation_assets/training_curves.png)

## Style transfer at inference time

To transfer a recipe to the opposite cuisine:

1. Encode the recipe → take its **content latent** (the cooking steps).
2. Discard its own style latent — glue the content latent to the **target cuisine's average style vector** instead.
3. Decode → a version of the same recipe, written in the target cuisine's style.

Decoding is free-running (no teacher forcing) with **no-repeat-trigram blocking**, since greedy decoding without it tends to loop ("stir stir stir stir...").

## A real example

Loaded live from `data/style_transfer_results.csv` (`02_model.ipynb`'s saved evaluation output) — not hand-copied text:

In [1]:
import pandas as pd

results = pd.read_csv('data/style_transfer_results.csv', index_col=0)

for name in ["bhindi bhaji", "agnello alla fiorentina (lamb florentine style)"]:
    row = results[results['recipe_name'] == name].iloc[0]
    print(f"{row['recipe_name']}  ({row['original_cuisine']} -> {row['target_cuisine']}, content F1={row['content_preservation_f1']:.2f})")
    print('ORIGINAL:   ', row['original_text'])
    print('TRANSFERRED:', row['generated_text'])
    print()


bhindi bhaji  (Indian -> Italian, content F1=0.21)
ORIGINAL:    wash bhindi in water and dry it using cloth or paper napkin. if possible, complete this process 2-3 hours prior to cooking. remove head and tail and chop it into 1/3-inch thick round circles.heat oil in a non-stick pan or heavy based kadai over medium flame. add cumin seeds and when they begin to crackle, add chopped garlic. sauté for 30 seconds.add chopped bhindi and mix well. cook on medium-low flame until bhindi turns dark green and shrinks. it will take approximately 6-8 minutes. stir in between occasionally.add chopped tomatoes, turmeric powder and salt; cook until tomatoes turn tender, approximately 2 minutes. add red chilli powder, garam masala powder and coriander powder; mix well. cook for a minute over low flame and turn off the flame. transfer prepared bhindi bhaji to a serving bowl. garnish with coriander leaves and serve with roti or paratha.
TRANSFERRED: if using wooden spoon or a or a knife or a shallow dish

## Evaluation methodology

Two **independent** checks (never the model's own training classifiers, to avoid grading its own homework):

- **Transfer strength** — a brand-new classifier, trained separately, guesses the cuisine of the *transferred* text. Agreement with the target cuisine = style actually changed.
- **Content preservation** — bag-of-words word-presence **F1** between the original recipe's content words and the transferred recipe's content words. High = the cooking steps survived; low = content got lost/altered too.

## Results

Held-out test set, best checkpoint (`checkpoint_best_recon.pt`):

| Metric | Value |
|---|---|
| Test reconstruction loss | 6.02 |
| Style classifier accuracy (style → cuisine) | 93.6% |
| Adversarial probe accuracy (content → cuisine, chance = 50%) | 53.0% |
| Transfer strength (independent classifier agrees with target style) | 95.0% |
| Content preservation (BoW F1, original vs. transferred) | 0.25 |

Real confusion matrices from `02_model.ipynb`:

![Confusion matrices: style_classifier vs adv_style_classifier](presentation_assets/confusion_matrices.png)

The adversarial probe sitting near chance (53% vs. 50%) means cuisine barely leaks into the content latent — the disentanglement is working. Transfer strength of 95% means the style change reads as convincing. **Content preservation at 0.25 is our main weak point.**

## Ablation: does the adversarial loss matter?

| | Adversarial probe accuracy (chance = 0.5) | Content preservation F1 |
|---|---|---|
| With adversarial loss (main model) | 0.53 | 0.25 |
| Without adversarial loss | 0.94 | 0.37 |

Real generated figure from `02_model.ipynb`:

![Ablation: adversarial loss on vs off](presentation_assets/ablation_comparison.png)

Turning the adversarial loss **off** lets cuisine leak heavily into content (0.94 vs. chance) — confirming the mechanism is doing real work. But it comes at a cost: content preservation is *better* without it (0.37 vs. 0.25), a genuine disentanglement-vs-fidelity tradeoff.

## Trying to fix content preservation

Content preservation (0.25 F1) was our main weak point. We bundled five candidate fixes into one retrained model to see if any of them moved that number:

1. **Attention** over per-token encoder states, instead of decoding from a single fixed content vector.
2. **A copy/pointer mechanism** — let the decoder directly copy a word from the source recipe instead of only generating from the vocabulary.
3. **A bigger content latent** (448 → 704 dims) — more room for content to avoid being squeezed.
4. **Retuned adversarial loss weight** (2.0 → 1.0) — trading some disentanglement strength for content fidelity.
5. **Teacher-forcing decay** (fixed 0.5 → linear 0.9 → 0.1) — weaning the model off ground-truth previous tokens during training, matching how it's actually used at inference.

## Attempt 1: it looked like a huge win — until we checked the text

| Metric | Baseline | All 5 changes |
|---|---|---|
| Content preservation F1 | 0.25 | **0.91** |
| Transfer strength | 0.96 | **0.03** |
| Adversarial probe acc (chance = 0.5) | 0.53 | 0.57 |

![Attempt 1 comparison: content preservation jumps but transfer strength collapses](presentation_assets/content_preservation_experiment_comparison.png)

**"roasted red bell pepper butter" (Indian → Italian):**

*Original:* "cut bell pepper in half. grill, turning oregularly, until the skin is blackened. remove from heat. place in a plastic sandwich bag for 15 minutes..."

*"Transferred":* "cut bell pepper in half grill turning and until the skin is blackened remove from heat place in a plastic sandwich bag for 15 minutes..."

**Why:** the copy mechanism can directly copy a word from the source recipe at each decode step. But this is an *autoencoder* — the reconstruction target *is* the source recipe — so the model found a trivial shortcut: "echo my own same-position source word." That's not real content understanding, and it explains why transfer strength collapsed: once the model relies on copying, the target style vector barely matters anymore, regardless of which cuisine it's given.

## Attempt 2: drop the copy mechanism — a real gain, still not free

| Metric | Baseline | Attention only, no copy |
|---|---|---|
| Content preservation F1 | 0.25 | **0.45** |
| Transfer strength | 0.96 | **0.49** |
| Adversarial probe acc (chance = 0.5) | 0.53 | **0.72** |

This time the improvement is real — the transferred text is genuinely different from the source, not an echo. But transfer strength still dropped to essentially a coin flip, and cuisine now leaks heavily into the content latent.

![Attempt 2 comparison: content preservation improves for real, but transfer strength and disentanglement both suffer](presentation_assets/content_preservation_nocopy_comparison.png)

**Why:** cuisine and content aren't fully independent in this dataset to begin with — our own class-imbalance analysis showed cuisine is ~98% separable from word choice alone (cumin, masala vs. parmesan, basil). A bigger content latent gives more room to retain that correlation, and a halved adversarial weight gives less pressure pushing it back out. Both changes were chosen to help content fidelity — and they did — but the mechanism they used to do it is hard to separate, in this data, from also retaining cuisine information.

## Takeaway

- Content preservation remains an open problem — neither attempt produced a clean win.
- Attempt 1's gain was a degenerate shortcut (a copy-mechanism artifact); Attempt 2's gain was real but came at the cost of the thing that made the original model work.
- The honest finding: content preservation is genuinely entangled with the adversarial disentanglement mechanism, not a simple architecture fix — a useful negative result, not a dead end.

## Challenges & struggles

- **Content preservation is genuinely weak (F1 0.25)** — we tested the capacity-limitation hypothesis directly (see the two follow-up experiments above): it turned out to be more specifically entangled with the adversarial disentanglement mechanism than a simple capacity fix.
- **Greedy decoding loops** during free-running generation (e.g. "the the the...") — fixed with no-repeat-trigram blocking, plus masking the `<UNK>` token.
- **Checkpoint compatibility across reruns** — retraining Word2Vec/BoW shifts vocabulary indices, which would silently break a saved checkpoint. Solved by only rebuilding these artifacts if missing.
- **Two-track tokenization** was needed — one column for both purposes would have hurt either fluency or the content signal.

## Conclusion & future work

- The disentanglement approach works well for the **style** side: 93.6% style classifier accuracy, 95% independent transfer strength, adversarial probe near chance.
- **Content preservation is the clear next target** — we tried a bigger content latent, attention over encoder states, and a lighter adversarial weight; the gains were either a training-shortcut artifact or came at the cost of transfer strength. The real fix likely needs to address the cuisine/content correlation in the data itself, not just add capacity.
- Class imbalance handling turned out not to matter much for this dataset — worth checking early on any project, since it can save time better spent elsewhere.

## Questions

Thank you!